# VadCLIP + CNN visual memory (UCF-Crime)

B?t GPU v? Internet. Th?m feature CLIP UCF-Crime `.npy` ?? gi?i n?n v?o Input. Notebook d?ng m? t? GitHub; ZIP `vadclip_cnn_memory_20261022.zip` l? c?ch chuy?n m? t?y ch?n. N?u ?? c? repo Kaggle c?, c?p nh?t b?ng `git pull --ff-only origin main` tr??c khi ch?y. ??y l? th? nghi?m, ch?a c? AUC x?c nh?n. Kh?ng thay file trong `src/`.


In [ ]:
from pathlib import Path
import os
import subprocess
import shutil
import sys
import zipfile

repo = Path('/kaggle/working/VadCLIP')
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Belldenchoi/VadCLIP.git', str(repo)], check=True)
os.chdir(repo)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ftfy', 'regex'], check=True)

# src/clip/clip.py uses the legacy pkg_resources.packaging import.
compat = subprocess.run([sys.executable, '-c', 'from pkg_resources import packaging'],
                        capture_output=True, text=True)
if compat.returncode:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'setuptools==69.5.1'], check=True)

bundles = list(Path('/kaggle/input').rglob('vadclip_cnn_memory_20261022.zip'))
if len(bundles) > 1:
    raise RuntimeError('More than one experiment ZIP found; keep one version')
if bundles:
    with zipfile.ZipFile(bundles[0]) as bundle:
        for name in bundle.namelist():
            if not name.startswith('experiments/cnn_memory/') or '..' in Path(name).parts:
                raise RuntimeError(f'Unexpected bundle entry: {name}')
        bundle.extractall(repo)
elif (repo / 'experiments/cnn_memory/cnn_memory_model.py').is_file():
    print('Using experiment code from the GitHub checkout')
else:
    # Kaggle may expose uploaded ZIP contents as an already extracted dataset.
    sources = [p.parent for p in Path('/kaggle/input').rglob('cnn_memory_model.py')
               if (p.parent / 'cnn_memory.py').is_file() and (p.parent / 'train.py').is_file()]
    if len(sources) != 1:
        raise RuntimeError(f'Update the repo with git pull --ff-only origin main, or add one experiment ZIP; found {len(sources)} folders')
    shutil.copytree(sources[0], repo / 'experiments/cnn_memory', dirs_exist_ok=True)
print('Experiment ready:', repo / 'experiments/cnn_memory')


N?u ?? c? hai CSV Kaggle ??ng ???ng d?n v? ??ng th? t? test, ??t ch?ng ? cell sau. N?u ch?a c?, ?? `REMAP_CSV = True`. C? nhi?u phi?n b?n feature th? thu h?p `FEATURE_ROOT` t?i ??ng m?t b?.


In [ ]:
REMAP_CSV = True
FEATURE_ROOT = '/kaggle/input'
if REMAP_CSV:
    subprocess.run([sys.executable, 'experiments/cnn_memory/prepare_csv.py',
                    '--feature-root', FEATURE_ROOT], check=True)
    TRAIN_CSV = 'outputs/cnn_memory_data/train.csv'
    TEST_CSV = 'outputs/cnn_memory_data/test.csv'
else:
    TRAIN_CSV = '/kaggle/working/ucf_CLIP_rgb_kaggle.csv'
    TEST_CSV = '/kaggle/working/ucf_CLIP_rgbtest_kaggle.csv'

# Doi OUTPUT khi doi run; giu cung seed/CSV cho control va run moi.
OUTPUT = 'outputs/cnn_memory_both_k3_s234'
BRANCH = 'both'  # 'none' = control; hoac 'c', 'a'
KERNEL = 3       # 1 hoac 3
SEED = 234
train_command = [sys.executable, 'experiments/cnn_memory/train.py',
                 '--train-list', TRAIN_CSV, '--test-list', TEST_CSV,
                 '--aux-branch', BRANCH, '--cnn-kernel', str(KERNEL),
                 '--seed', str(SEED), '--output-dir', OUTPUT]
subprocess.run(train_command + ['--preflight'], check=True)


Train 10 epoch m?c ??nh. Nh?nh CNN b?t t? epoch 3; `best.pt` ch?n theo test AUC1 t? epoch 3, `last.pt` d?ng resume. C? hai nh?nh c? loss MIL g?c; th?m loss memory khi nh?nh ph? ho?t ??ng.


In [ ]:
subprocess.run(train_command, check=True)


In [ ]:
subprocess.run([sys.executable, 'experiments/cnn_memory/evaluate.py',
                '--test-list', TEST_CSV, '--checkpoint', f'{OUTPUT}/best.pt',
                '--log-path', f'{OUTPUT}/evaluate.log'], check=True)


K?t qu? n?m trong th? m?c OUTPUT: `train.log`, `metrics.jsonl`, `config.json`, `best.pt`, `last.pt`. L?u to?n b? th? m?c tr??c khi k?t th?c session. ?? ??i chi?u, ??i BRANCH th?nh `none`, ??i OUTPUT sang t?n ri?ng v? ch?y l?i train v?i c?ng seed. Mu?n resume: d?ng l?i ??ng t?y ch?n c? v? th?m `--resume <OUTPUT>/last.pt` v?o train_command.
